# 第2章 机器学习基础 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter2/Chapter2.md]]
- [[课程/深度学习/课堂笔记/2026-09-02-第2节-机器学习基础概念与模型构建教学.md]]
- [[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]
- [[课程/深度学习/演示代码/Chapter2.py]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## T2-B 固定八次模型：用验证集选择正则化

**来源与性质**：[[课程/深度学习/PPT/Chapter2/Chapter2.md]]正则化例2与验证集；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-03；[[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]正则化补充推导。新编换目标、换选型维度的模拟题。


**100分，30分钟，一道完整题。** 从导入开始编写。本题不重新选择次数，也不重拟合训练+验证数据，区别于测试A。

1. **数据（15分）**：`rng=np.random.default_rng(20261202)`；依次生成70个 `[-1,1)` 横坐标 `ridge_b_x`、70个 `[-.06,.06)` 均匀噪声；标签 `ridge_b_y=exp(-2*x*x)+0.2*x+noise`。前14点训练、接16点验证、后40点测试，所有误差对带噪标签计算。
2. **正则拟合函数（30分）**：实现 `ridge8(x,y,lam)`，返回固定8次多项式的9个升幂系数，优化 `mean((A@w-y)**2)+lam*sum(w**2)`，全部系数包括常数项均惩罚。用增广最小二乘，不显式求逆；lam=0为普通最小二乘，lam<0抛ValueError。写明增广下部为什么是 `sqrt(N*lam)`。
3. **验证选型（25分）**：候选lambda为 `[0,1e-5,1e-3,1e-1]`。每组只用14训练点拟合，保存 `ridge_b_coefs` 字典（lambda为键）与 `ridge_b_table`（4行，列为lambda、训练RMSE、验证RMSE、系数范数）。验证RMSE最小者获选，恰好同分选较小lambda，保存 `ridge_b_chosen`。**选型后保留该14点模型，不合并验证重拟合**。
4. **最终测试（15分）**：保存 `ridge_b_final_coef,ridge_b_test_pred,ridge_b_train_rmse,ridge_b_test_rmse,ridge_b_gap_abs,ridge_b_gap_signed`；绝对差为课件泛化差距，有符号差额外输出。打印列名、选中lambda和误差，测试不能反向改变选择。
5. **图与解释（15分）**：画真实函数、选中模型、训练／验证／测试点，创建本地输出目录后保存 `output/Q201B.png`；解释过强正则化为何可能欠拟合、为何不能把带惩罚总目标当作RMSE。

`rcond=None` 必须显式给出；代表性数值容差 `atol=1e-8`。图及解释人工核对。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
def ridge8(x,y,lam):
    if lam<0:
        raise ValueError('negative lambda')
    A=np.column_stack([x**j for j in range(9)])
    if lam==0:
        return np.linalg.lstsq(A,y,rcond=None)[0]
    n=x.size; B=np.zeros((n+9,9)); z=np.zeros(n+9)
    B[:n]=A; z[:n]=y
    for j in range(9): B[n+j,j]=np.sqrt(n*lam)
    return np.linalg.lstsq(B,z,rcond=None)[0]
_rng=np.random.default_rng(20261202)
ridge_b_x=_rng.uniform(-1,1,70)
ridge_b_y=np.exp(-2*ridge_b_x**2)+.2*ridge_b_x+_rng.uniform(-.06,.06,70)
ridge_b_coefs={}; _rows=[]
_A=np.column_stack([ridge_b_x[:14]**j for j in range(9)])
_Av=np.column_stack([ridge_b_x[14:30]**j for j in range(9)])
for _lam in [0.,1e-5,1e-3,1e-1]:
    _c=ridge8(ridge_b_x[:14],ridge_b_y[:14],_lam); ridge_b_coefs[_lam]=_c
    _er=float(np.sqrt(np.mean((_A@_c-ridge_b_y[:14])**2)))
    _ev=float(np.sqrt(np.mean((_Av@_c-ridge_b_y[14:30])**2)))
    _rows.append([_lam,_er,_ev,np.linalg.norm(_c)])
ridge_b_table=np.array(_rows)
ridge_b_chosen=min(_rows,key=lambda r:(r[2],r[0]))[0]
ridge_b_final_coef=ridge_b_coefs[ridge_b_chosen]
_At=np.column_stack([ridge_b_x[30:]**j for j in range(9)])
ridge_b_test_pred=_At@ridge_b_final_coef
ridge_b_train_rmse=float(np.sqrt(np.mean((_A@ridge_b_final_coef-ridge_b_y[:14])**2)))
ridge_b_test_rmse=float(np.sqrt(np.mean((ridge_b_test_pred-ridge_b_y[30:])**2)))
ridge_b_gap_signed=ridge_b_test_rmse-ridge_b_train_rmse
ridge_b_gap_abs=abs(ridge_b_gap_signed)
print('lambda train14_RMSE validation16_RMSE coefficient_norm',ridge_b_table,sep='; ')
print('selected lambda:',ridge_b_chosen)
print('train14_RMSE test40_RMSE gap_abs gap_signed')
print(ridge_b_train_rmse,ridge_b_test_rmse,ridge_b_gap_abs,ridge_b_gap_signed)
Path('output').mkdir(exist_ok=True)
plt.figure(); _g=np.linspace(-1,1,201)
plt.plot(_g,np.exp(-2*_g**2)+.2*_g,'k--',label='target')
plt.plot(_g,np.column_stack([_g**j for j in range(9)])@ridge_b_final_coef,label='selected ridge')
for _a,_b,_name in [(0,14,'train'),(14,30,'validation'),(30,70,'test')]:
    plt.plot(ridge_b_x[_a:_b],ridge_b_y[_a:_b],'.',label=_name)
plt.xlabel('x'); plt.ylabel('observed y'); plt.legend(); plt.savefig('output/Q201B.png'); plt.close()
print('Multiplying the mean objective by N produces the N*lambda penalty in squared-sum form.')
print('Excessive regularization can suppress useful coefficients and underfit.')
print('The penalized objective contains an extra term; it is not RMSE.')


### 解析与点评

A选择容量并重拟合，B固定容量、选择惩罚强度并保留原训练模型；不是仅换seed。N是当前拟合样本数14，不能错用70或验证集大小。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
_x=np.array([-1.,0.,1.]); _y=np.array([2.,1.,6.])
_w=ridge8(_x,_y,.2); _A=np.column_stack([_x**j for j in range(9)])
assert np.linalg.norm(_A.T@(_A@_w-_y)+3*.2*_w)<1e-9
assert np.allclose(_A@ridge8(_x,_y,0),_y,atol=1e-8)
try: ridge8(_x,_y,-.1)
except ValueError: pass
else: raise AssertionError('负lambda应拒绝')
_rng=np.random.default_rng(20261202); _x=_rng.uniform(-1,1,70)
_y=np.exp(-2*_x**2)+.2*_x+_rng.uniform(-.06,.06,70)
assert np.allclose(ridge_b_x,_x) and np.allclose(ridge_b_y,_y)
_check_ridge_b_table = np.asarray(ridge_b_table, dtype=float)
assert np.allclose(_check_ridge_b_table[:,0],[0,1e-5,1e-3,1e-1])
_A=np.column_stack([_x[:14]**j for j in range(9)])
_Av=np.column_stack([_x[14:30]**j for j in range(9)])
for _i,_lam in enumerate([0.,1e-5,1e-3,1e-1]):
    _c=ridge_b_coefs[_lam]
    assert np.linalg.norm(_A.T@(_A@_c-_y[:14])+14*_lam*_c)<1e-8
    assert np.isclose(_check_ridge_b_table[_i,2],np.sqrt(np.mean((_Av@_c-_y[14:30])**2)))
    assert np.isclose(_check_ridge_b_table[_i,3],np.linalg.norm(_c))
_best=min(_check_ridge_b_table.tolist(),key=lambda r:(r[2],r[0]))
assert ridge_b_chosen==_best[0]
assert np.allclose(ridge_b_final_coef,ridge_b_coefs[ridge_b_chosen],atol=1e-8)
_At=np.column_stack([_x[30:]**j for j in range(9)])
assert np.allclose(ridge_b_test_pred,_At@ridge_b_final_coef,atol=1e-8)
assert np.isclose(ridge_b_train_rmse,np.sqrt(np.mean((_A@ridge_b_final_coef-_y[:14])**2)))
assert np.isclose(ridge_b_test_rmse,np.sqrt(np.mean((ridge_b_test_pred-_y[30:])**2)))
assert np.isclose(ridge_b_gap_abs,abs(ridge_b_test_rmse-ridge_b_train_rmse))
assert np.isclose(ridge_b_gap_signed,ridge_b_test_rmse-ridge_b_train_rmse)
assert Path('output/Q201B.png').stat().st_size>1000

print("自检通过")
